# SQL Basics
Before use this notebook, you are encouraged to go over basic SQL syntax in  __[Open Geospatial Solutions](https://geog-414.gishub.org/)__ under DuckDB section. 

This notebook demonstrates a simple workflow: __Find population density for Intermediate Zones in Edinburgh__. 

The purposes are 
* access SIMD and Datazone lookup files
* aggregating the population from Datazones to Intermediate Zones
* join population to Intermediate Zones shapefile
* export as geojson for future use 
  
## Installation

DuckDB is extremely easy to install. Go to __[https://duckdb.org/#quickinstall](https://duckdb.org/#quickinstall)__ and select the programming language you are using. 

Due to some version issues with GeoPandas, I recommend following these steps to create a conda environment for this exercise.
<div class="alert alert-block alert-danger">
<b>Note:</b> conda install GeoPandas before DuckDB and others!
</div>

1. conda create -n "geo_env" python=3.13.1
2. conda activate geo_env
3. conda config --env --add channels conda-forge
4. conda config --env --set channel_priority strict
5. conda install python=3 geopandas
6. conda install jupyter

If you are keen to develop your skill in SQL, worth to try a SQL IDE. We recommend DBeaver which can be installed here __[DBeaver](https://dbeaver.io/)__ (use community version). 
This is not necessary for this class.

## More SQL samples are included in
### /SpatialSQLBasics/


## Let's start 

Uncomment the following cell to install the required packages.



In [ ]:
# %pip install duckdb duckdb-engine jupysql leafmap

In [ ]:
import duckdb
import pandas as pd

# Import jupysql Jupyter extension to create SQL cells
%load_ext sql


Connect to jupysql to DuckDB using SQLAlchemy-style connection string. %sql for one line SQL, %%sql for multiline SQL.<br>

One of the advantages of DuckDB is its simplicity.<br>

As you notice here it can be connected to a db which can be stored in your hard drive or an in-memory db. <br>

The former is just a file ending with .db no other overheads. Extremely portable. <br>

The latter stores everything in your memory meaning it is very fast but every time you re-start the kernel you lose the db. <br>

No right or wrong. Choose what your need. <br>


In [ ]:
# %sql duckdb:///:memory:
%sql duckdb:///SQL2.db

Use **'Select * from <file, Table, Dataframe>'** to explore what's inside first.

We now access the SIMD csv file which comes with your SIMD downloads.

You can find the file the Lab9 folder on GitHub. 
Download it to your DuckDB local folder to avoid any http block issue. 

DuckDB uses httpfs extension to direcly access to remote files. 
Try it  when you are free. 

In [ ]:
%%sql
Select * from 'Data\simd2020_withinds.csv'


In [ ]:
# %%sql

# INSTALL httpfs;
# LOAD httpfs;

In [ ]:
# %%sql
# Select * from read_csv('https://your online file.csv');

Similarly, we explore the lookup file.

We can find that **SIMD** file has a **Intermediate_Zone** field but no Intermediate Zone Code. 
Population is at **Data_Zone** level. One **Intermediate_Zone** can have multiple **Data_Zone**.

While the **lookup** file have **Intermediate_Zone** name in **IZ2011_Name** field with additional information such as **IZ2011_Code**, **LA_code**. 

Is there other potential link between these two tables?

In [ ]:
%%sql

Select * from 'Data\DataZone2011lookup.csv'; 
        


I create copies of these to csvs to my local DuckDB.
The SQL used here is **'Create Table [Table Name] as Select * from <file, Table, Dataframe>'**. 

In [ ]:
%%sql 

CREATE TABLE simd AS SELECT * FROM 'Data\simd2020_withinds.csv';
CREATE TABLE lookup AS SELECT * FROM 'Data\DataZone2011lookup.csv';


After creating the tables in DuckDB. I tried to show them.
Instead of using *Select*, I used **'From [Table Name] limit x'**> 
This is somethine nice not normally used in other DBs. 
Note you can limit number of records you want to see. 

In [ ]:
%%sql 

From simd limit 5;

In [ ]:
%%sql

from lookup limit 3;

Often, we what to know the scale of the dataset in addition to number of records.
Try to use **Select Distinct [FieldName]  from [TableName]**. 
You can also use **count distinct** to check the total numbers

In [ ]:
%%sql

SELECT DISTINCT Intermediate_Zone FROM simd LIMIT 10;

In [ ]:
%%sql

SELECT COUNT(DISTINCT Intermediate_Zone) FROM simd;

If you feel this is easy enough, now we are ready to write slightly harder SQLs. They make things a bit complicated in either **Select**, **from**, **where** or add **group by** or sort. </br>
Remember your aggregation functions normally happen within **Select**, such as Sum, Avg. </br>
**where** clause is used for filtering. Notice we use wildcard instead of an exact match for strings. </br>
It has to be worked together with **group by** where your study unit becomes larger, such as from Datazone to Intermediate zones. </br>
Adjust **order by** to sort the results if needed.</br>

The following SQL calculates the sum of the population, originally at the Datazone level, to Intermediate Zones in Edinburgh.


In [ ]:
%%sql

SELECT SUM(Total_population) as pop,Intermediate_Zone, Council_area FROM simd where Council_area like '%Edinburgh' group by Intermediate_Zone,Council_area order by pop DESC, Council_area ASC;

This structure remains even if we add in more conditions. 


In [ ]:
%%sql

SELECT SUM(Total_population) as pop,Intermediate_Zone, Council_area FROM simd where Council_area like '%Edinburgh' Or Council_area like '%Glasgow%' group by Intermediate_Zone,Council_area having Sum(Total_population)>5000 order by pop ASC, Council_area ASC; 

It is fine to look at the selection results but it would be also useful to create a table out from it for further use. </br>
We add **Create Table [new table name] as**. </br>
In more complicated queries known as transactions, you can create table views rather than tables.  </br>
For DuckDB, we can create very big tables without too much concern about the storage due to its unique structure.  </br>

In [ ]:
%%sql

create table if not exists Edinburgh_pop as (
    SELECT SUM(Total_population) as pop,Intermediate_Zone, Council_area FROM simd where Council_area like '%Edinburgh' group by Intermediate_Zone,Council_area order by pop DESC, Council_area ASC
);

You can check out if the new table is created successively. </br>
Also you can view its details. 

In [ ]:
%%sql SHOW TABLES;

SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_type = 'BASE TABLE';


In [ ]:
%%sql

select * from Edinburgh_pop limit 3;

After aggregating the population to Intermediate Zones, you can join it to the lookup file where we have additional Intermediate Zone code information. </br>
The syntax here is **Join** clause after **from**. Note that you need to specify the matching fields between the two tables. </br>
If the matching fields are specifically created to set up *relationship* between tables also maintain *integrity*, they are called *Keys* or specifically *Foreign keys*. </br>
In our case, we just use Intermediate Zone Names. They are not set up as *Keys* but function the same.</br>

Meanwhile, we can see show different Join method works. Change **Left Join** to **Right Join**, **Inner Join** and **Full Join** to see the difference. 


In [ ]:
%%sql

With Edinburgh_pop_IZCode as(
    SELECT pop, Edinburgh_pop."Intermediate_Zone", Edinburgh_pop."Council_area", lookup."IZ2011_Code" 
    FROM Edinburgh_pop Left JOIN lookup ON Edinburgh_pop.Intermediate_Zone = lookup."IZ2011_Name"
)
select count(distinct IZ2011_Code) from Edinburgh_pop_IZCode;

------

In [ ]:
%sql duckdb:///SQL.db
%sql disconnect


We have got it worked! </br>
But there is still a problem. </br>

Edinburgh has 111 Intermediate Zones in total. But our SQL returns 113 instead.</br>

In your exercises, you will fix the SQL to retrieve correct Intermediate Zones and try to explain the issue. 

Hint: Check matching between Intermdiate Zone Names in SIMD and Lookup table.